# Bab 12
# Neural Networks That Write Like Shakespeare

**Nama:** Omar Al Farizy Hutagalung  
**NIM:** 101032300034  
**Mata Kuliah:** Deep Learning

## 1. Tujuan Bab

Bab ini membahas bagaimana jaringan saraf memproses urutan teks dan memanfaatkan konteks sebelumnya. Berbeda dari klasifikasi teks yang langsung menghasilkan satu label, pemodelan urutan perlu mempertimbangkan urutan elemen dan hubungan antarposisi.

Fokus pembelajaran:
1. Memahami teks sebagai urutan token.
2. Memahami sentence embeddings dan representasi numerik kalimat.
3. Memahami transition matrices sebagai representasi hubungan antar-token.
4. Mengikuti proses forward propagation pada jaringan berulang (*recurrent neural network*, RNN).
5. Memahami pengaruh hidden state dan urutan masukan.
6. Memahami tantangan urutan dengan panjang yang bervariasi.
7. Membedakan demonstrasi konsep dari reproduksi eksperimen buku.

**Catatan:** kode pada notebook ini adalah demonstrasi NumPy yang bisa dijalankan secara mandiri. Kode ini tidak diklaim sebagai salinan persis listing program di buku.

## 2. Teks sebagai data berurutan

Teks bukan sekumpulan kata yang tidak berurutan. Susunan token memengaruhi makna. Misalnya, kalimat “anjing mengejar kucing” berbeda makna dari “kucing mengejar anjing”, walaupun keduanya menggunakan kata yang sama.

Agar teks dapat diproses jaringan saraf, langkah awal umumnya adalah:
1. memecah teks menjadi token (misalnya kata atau karakter);
2. membangun vocabulary;
3. mengubah token menjadi indeks numerik;
4. menyusun indeks menjadi urutan yang dapat diproses model.

Indeks token hanya menunjukkan identitas token di dalam vocabulary. Jarak numerik antarindeks tidak menyatakan kedekatan makna.

In [ ]:
sentences = [
    "the cat chases the mouse",
    "the mouse chases the cat",
    "the cat sleeps",
]

tokens = [sentence.split() for sentence in sentences]
vocabulary = sorted({word for sentence in tokens for word in sentence})
word_to_id = {word: i for i, word in enumerate(vocabulary)}
id_to_word = {i: word for word, i in word_to_id.items()}

encoded = [[word_to_id[word] for word in sentence] for sentence in tokens]

print("Vocabulary:", vocabulary)
for sentence, ids in zip(sentences, encoded):
    print(sentence, "->", ids)

## 3. Sentence embeddings

**Sentence embedding** adalah representasi numerik untuk sebuah kalimat atau urutan teks. Representasi ini memungkinkan model menggunakan operasi matematis untuk memproses informasi bahasa.

Salah satu demonstrasi sederhana adalah merata-ratakan vektor embedding kata. Cara ini mudah dipahami, tetapi mengabaikan urutan kata. RNN menggunakan pendekatan berbeda: representasi diperbarui secara bertahap saat token berikutnya diproses.

Perbedaan penting:
- **Word embedding:** representasi untuk sebuah kata/token.
- **Sentence/sequence representation:** representasi yang merangkum informasi dari beberapa token.
- **Hidden state RNN:** representasi yang diperbarui dari satu langkah urutan ke langkah berikutnya.

Representasi kalimat tidak otomatis menyimpan seluruh makna bahasa. Kualitasnya bergantung pada data, objective pelatihan, dan arsitektur yang digunakan.

In [ ]:
import numpy as np

rng = np.random.default_rng(12)
embedding_dim = 4
embeddings = {
    word: rng.normal(size=embedding_dim)
    for word in vocabulary
}

def mean_sentence_embedding(sentence):
    vectors = [embeddings[word] for word in sentence.split()]
    return np.mean(vectors, axis=0)

for sentence in sentences:
    print(sentence)
    print(np.round(mean_sentence_embedding(sentence), 3))

### Keterbatasan rata-rata embedding

Rata-rata embedding memberi representasi dengan panjang tetap, tetapi urutan kata tidak dipertahankan secara eksplisit. Kalimat dengan kumpulan kata yang sama dapat memperoleh representasi rata-rata yang sama meskipun susunannya berbeda.

Untuk tugas yang bergantung pada urutan, model dapat memproses token satu per satu dan menyimpan ringkasan konteks melalui hidden state. Inilah salah satu motivasi utama RNN.

## 4. Transition matrices dan hubungan antar-token

Transition matrix dapat digunakan untuk merangkum pola token yang cenderung muncul setelah token lain. Dalam contoh sederhana, setiap elemen matriks menghitung transisi dari satu kata ke kata berikutnya pada data.

Matriks transisi yang dihitung langsung dari frekuensi bukan RNN terlatih. Namun, contoh ini memperlihatkan bagaimana struktur urutan dapat diringkas secara numerik.

In [ ]:
from collections import defaultdict

counts = np.zeros((len(vocabulary), len(vocabulary)), dtype=float)

for sentence in encoded:
    for current_id, next_id in zip(sentence[:-1], sentence[1:]):
        counts[current_id, next_id] += 1

row_sums = counts.sum(axis=1, keepdims=True)
transition_probs = np.divide(
    counts,
    row_sums,
    out=np.zeros_like(counts),
    where=row_sums != 0
)

print("Transition probability matrix shape:", transition_probs.shape)
print("Kolom/indeks vocabulary:", vocabulary)

for word in vocabulary:
    i = word_to_id[word]
    if counts[i].sum() > 0:
        next_idx = int(np.argmax(transition_probs[i]))
        print(f"Setelah '{word}', transisi paling sering dalam data contoh: "
              f"'{id_to_word[next_idx]}'")

### Interpretasi transition matrix

- Baris mewakili token saat ini.
- Kolom mewakili token berikutnya.
- Nilai pada matriks probabilitas menunjukkan frekuensi relatif transisi yang teramati.
- Jika suatu kata tidak pernah memiliki token lanjutan di data contoh, barisnya berisi nol.

Matriks ini bergantung kuat pada korpus. Korpus kecil menghasilkan estimasi yang tidak stabil dan tidak dapat mewakili bahasa secara umum. Model bahasa neural mempelajari parameter melalui objective pelatihan, bukan sekadar menyalin frekuensi transisi yang terlihat.

## 5. Recurrent Neural Network (RNN)

RNN memproses satu elemen urutan pada setiap langkah. Informasi dari langkah sebelumnya dirangkum dalam hidden state dan digunakan ketika memproses token berikutnya.

Bentuk sederhana persamaan forward propagation:

\[
h_t = \tanh(W_x x_t + W_h h_{t-1} + b_h)
\]

Keterangan:
- \(x_t\): representasi token pada waktu/posisi \(t\);
- \(h_{t-1}\): hidden state dari langkah sebelumnya;
- \(W_x\): bobot input ke hidden state;
- \(W_h\): bobot rekuren dari hidden state sebelumnya;
- \(b_h\): bias;
- \(h_t\): hidden state yang baru.

Bobot yang sama digunakan pada setiap langkah urutan. Parameter bersama ini memungkinkan model menerapkan aturan pemrosesan yang sama pada token di posisi berbeda.

In [ ]:
def rnn_forward(sequence, Wx, Wh, bh, h0=None):
    hidden_size = Wh.shape[0]
    h = np.zeros(hidden_size) if h0 is None else h0.copy()
    states = []

    for x_t in sequence:
        h = np.tanh(Wx @ x_t + Wh @ h + bh)
        states.append(h.copy())

    return states

# Demonstrasi dengan embedding kata sebagai input.
input_size = embedding_dim
hidden_size = 5
rng = np.random.default_rng(120)

Wx = rng.normal(0, 0.3, size=(hidden_size, input_size))
Wh = rng.normal(0, 0.3, size=(hidden_size, hidden_size))
bh = np.zeros(hidden_size)

sample_sentence = "the cat chases the mouse"
sequence_vectors = [embeddings[word] for word in sample_sentence.split()]
states = rnn_forward(sequence_vectors, Wx, Wh, bh)

for token, state in zip(sample_sentence.split(), states):
    print(f"{token:8s} hidden state = {np.round(state, 3)}")

### Memahami hasil forward propagation

Setiap token menghasilkan hidden state. Hidden state tersebut dipengaruhi oleh:
1. representasi token saat ini;
2. hidden state sebelumnya;
3. parameter bersama pada RNN.

Karena hidden state diteruskan, model memiliki mekanisme untuk membawa informasi dari bagian sebelumnya. Akan tetapi, hidden state berukuran terbatas dan tidak menjamin seluruh informasi awal akan dipertahankan. RNN sederhana juga dapat mengalami kesulitan mempelajari dependensi jangka panjang.

## 6. Mengapa urutan penting?

Bandingkan dua urutan:
- `the cat chases the mouse`
- `the mouse chases the cat`

Kedua urutan menggunakan kumpulan token serupa, tetapi urutannya berbeda. RNN memproses token secara berurutan, sehingga perubahan urutan mengubah urutan operasi dan biasanya mengubah hidden state akhir.

Contoh berikut membandingkan representasi akhir dua urutan dengan parameter RNN yang sama.

In [ ]:
sentence_a = "the cat chases the mouse"
sentence_b = "the mouse chases the cat"

def encode_sequence(text):
    return [embeddings[word] for word in text.split()]

state_a = rnn_forward(encode_sequence(sentence_a), Wx, Wh, bh)[-1]
state_b = rnn_forward(encode_sequence(sentence_b), Wx, Wh, bh)[-1]

print("Hidden state akhir A:", np.round(state_a, 4))
print("Hidden state akhir B:", np.round(state_b, 4))
print("Sama persis?", np.allclose(state_a, state_b))
print("Jarak Euclidean:", round(float(np.linalg.norm(state_a - state_b)), 4))

Perbedaan hidden state pada contoh ini menunjukkan bahwa urutan dapat memengaruhi representasi. Karena bobot dibuat acak dan tidak dilatih, eksperimen ini hanya menunjukkan mekanisme komputasi; ia tidak membuktikan bahwa model memahami makna kalimat.

## 7. Urutan dengan panjang bervariasi

Kalimat memiliki jumlah token yang berbeda-beda. RNN dapat memproses urutan dengan panjang berbeda karena operasi rekuren dilakukan untuk setiap token yang tersedia. Hidden state akhir dapat dipakai sebagai ringkasan urutan dengan panjang tetap.

Dalam batch pelatihan, urutan dengan panjang berbeda sering perlu ditangani menggunakan padding dan masking, atau dikelompokkan berdasarkan panjang. Padding menambahkan token khusus agar tensor memiliki ukuran seragam; masking mencegah posisi padding diperlakukan sebagai konten biasa dalam perhitungan yang relevan.

Hal penting:
- padding adalah mekanisme penyelarasan bentuk tensor, bukan kata bermakna;
- panjang asli urutan tetap perlu diketahui;
- evaluasi dan loss harus menangani token padding secara tepat.

In [ ]:
variable_sentences = [
    "the cat sleeps",
    "the cat chases the mouse",
    "the mouse sleeps"
]

for text in variable_sentences:
    states = rnn_forward(encode_sequence(text), Wx, Wh, bh)
    print(f"{len(text.split())} token(s): {text}")
    print("  hidden state akhir:", np.round(states[-1], 3))

## 8. Dari representasi urutan ke prediksi bahasa

Untuk menghasilkan teks, model bahasa dapat memprediksi token berikutnya berdasarkan representasi konteks. Secara umum:
1. token saat ini direpresentasikan sebagai vektor;
2. RNN memperbarui hidden state;
3. lapisan keluaran menghasilkan skor untuk token kandidat;
4. skor diubah menjadi distribusi probabilitas;
5. token berikutnya dipilih dari distribusi tersebut atau dipilih berdasarkan skor tertinggi;
6. token baru menjadi bagian dari konteks langkah berikutnya.

Kualitas prediksi bergantung pada parameter yang telah dipelajari dari data. RNN dengan bobot acak seperti pada demonstrasi sebelumnya tidak akan menghasilkan teks yang bermakna.

In [ ]:
def softmax(scores):
    scores = scores - np.max(scores)
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum()

# Contoh head keluaran acak untuk menunjukkan bentuk distribusi token.
rng = np.random.default_rng(121)
output_weights = rng.normal(0, 0.2, size=(len(vocabulary), hidden_size))
output_bias = np.zeros(len(vocabulary))

final_state = rnn_forward(encode_sequence("the cat"), Wx, Wh, bh)[-1]
scores = output_weights @ final_state + output_bias
probabilities = softmax(scores)

top_ids = np.argsort(probabilities)[::-1][:3]
print("Tiga kandidat token dengan probabilitas tertinggi:")
for idx in top_ids:
    print(f"{id_to_word[idx]:10s} {probabilities[idx]:.3f}")

**Peringatan interpretasi:** lapisan keluaran di atas diinisialisasi acak, bukan dilatih. Probabilitas yang dicetak hanya menunjukkan bagaimana skor dapat diubah menjadi distribusi. Jangan menafsirkannya sebagai prediksi bahasa yang berkualitas.

## 9. Batasan RNN sederhana

RNN dasar memberikan cara langsung untuk membawa informasi antar-langkah, tetapi memiliki keterbatasan:
- hidden state memiliki kapasitas terbatas;
- informasi dari token yang jauh dapat menjadi sulit dipertahankan;
- pelatihan pada urutan panjang dapat menghadapi masalah gradien yang menghilang atau membesar;
- hasilnya bergantung pada data dan proses optimisasi.

Arsitektur lanjutan seperti LSTM dan GRU dikembangkan untuk membantu mengelola informasi jangka panjang. Bab ini menjadi dasar untuk memahami mengapa pemrosesan urutan memerlukan desain yang berbeda dari model feedforward biasa.

## 10. Kesimpulan Bab

Poin utama:
1. Bahasa merupakan data berurutan; urutan token memengaruhi makna dan prediksi.
2. Token perlu diubah menjadi representasi numerik.
3. Sentence embeddings merangkum teks, tetapi metode seperti rata-rata embedding dapat mengabaikan urutan.
4. Transition matrix merangkum frekuensi transisi token dalam data, tetapi tidak sama dengan RNN yang dilatih.
5. RNN memperbarui hidden state secara berurutan menggunakan bobot rekuren bersama.
6. Hidden state akhir dapat menjadi representasi urutan dengan panjang tetap.
7. Prediksi token berikutnya memerlukan model yang dilatih, bukan sekadar bobot acak.
8. Urutan dengan panjang berbeda dapat diproses, tetapi batching memerlukan perhatian terhadap padding dan masking.

## Pertanyaan refleksi
1. Mengapa rata-rata embedding kata tidak cukup untuk membedakan dua kalimat dengan kata sama tetapi urutan berbeda?
2. Apa peran hidden state pada RNN?
3. Apa perbedaan transition matrix berbasis frekuensi dan RNN yang dilatih?
4. Mengapa output dari model dengan bobot acak tidak boleh dianggap sebagai prediksi bahasa yang valid?
5. Apa fungsi padding dan masking saat melatih model dengan urutan berbeda panjang?